# Risk/UQ Paper Track: UQ Benchmark

## Objective
Evaluate calibration robustness and selective-safety behavior across nominal and shift suites using persisted training artifacts.


## Hypotheses Being Tested
- **H1:** calibrated monitor outperforms raw monitor on ECE in `nominal_clean`.
- **H2:** calibration benefits carry to most shift suites.
- **H3:** selective risk improves at equivalent coverage.


## Methodology
- Use persisted risk dataset + trained artifacts from training notebook.
- Generate calibrated predictions for test/holdout splits.
- Compute benchmark tables: calibration, discrimination, reliability, coverage-risk, shift gap.
- Resume behavior follows `RUN_MODE`.


## Step 1 - Bootstrap Environment

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/achyutmorang/waymax-simulation-experiments.git'
REPO_DIR = '/content/waymax-simulation-experiments'
REPO_BRANCH = 'main'
REQUIRED_DRIVE_FOLDER = '/content/drive/MyDrive/waymax_experiments'

runtime_cfg_overrides = {
    'verify_drive_access_every_run': False,
    'force_reinstall': False,
    'auto_restart_after_setup': True,
    'strict_lockfile_check': True,
    'setup_cache_enabled': True,
    'revalidate_core_imports_on_cache_hit': True,
    'setup_cache_path': '/content/.closedloop_setup_cache.json',
    'force_module_hot_reload': True,
}

content_root = Path('/content')
content_root.mkdir(parents=True, exist_ok=True)
try:
    _ = os.getcwd()
except FileNotFoundError:
    os.chdir(str(content_root))

if not Path(REPO_DIR).exists():
    subprocess.run(['git', 'clone', '--depth', '1', '-b', REPO_BRANCH, REPO_URL, REPO_DIR], check=True)

for path in (REPO_DIR, f'{REPO_DIR}/src'):
    if path not in sys.path:
        sys.path.insert(0, path)

from src.platform import ColabRuntimeConfig, bootstrap_colab_runtime_with_config

runtime_cfg = ColabRuntimeConfig(
    repo_url=REPO_URL,
    repo_dir=REPO_DIR,
    repo_branch=REPO_BRANCH,
    required_drive_folder=REQUIRED_DRIVE_FOLDER,
    **runtime_cfg_overrides,
)
bootstrap = bootstrap_colab_runtime_with_config(runtime_cfg)

globals()['_RISK_UQ_REPO_REV'] = str(bootstrap.repo_sync.repo_rev)
setup_result = bootstrap.setup.result

print('Working directory:', os.getcwd())
print('Repo commit:', bootstrap.repo_sync.repo_rev)
print('Drive mounted:', bootstrap.drive_status.mounted)
print('[setup] result:', setup_result)
if bool(setup_result.get('restart_required', False)) and (not bool(runtime_cfg.auto_restart_after_setup)):
    print('[setup] restart_required=True; restart runtime before continuing.')


## Step 2 - Configure Run Identity and Benchmark Knobs

In [ ]:
from src.workflows import initialize_run_context, report_run_context

# Leave RUN_TAG empty to auto-adopt latest matching run under PERSIST_ROOT.
RUN_TAG = ''
RUN_TAG_PREFIX = 'risk_uq'
PERSIST_ROOT = '/content/drive/MyDrive/waymax_experiments/risk_uq_runs/v1'
RUN_MODE = 'auto'  # auto | fresh | resume

run_context = initialize_run_context(
    run_tag=RUN_TAG,
    persist_root=PERSIST_ROOT,
    n_shards=1,
    shard_id=0,
    auto_run_main_loop_when_ready=False,
    run_main_loop_override=False,
    run_tag_prefix=RUN_TAG_PREFIX,
    planner_backend='latentdriver',
    planner_surprise_name='predictive_seq_w2',
    auto_generate_run_tag_if_empty=True,
    resume_mode=RUN_MODE,
    warn_on_config_drift=True,
)

cfg = run_context.cfg
search_cfg = run_context.search_cfg
run_prefix = run_context.run_prefix
print('run_prefix =', run_prefix)
report_run_context(run_context, display_fn=display)

# Risk/UQ experiment knobs
cfg.risk_dataset_build = True
cfg.risk_dataset_candidate_count = 8
cfg.risk_dataset_control_horizon_steps = 6
cfg.risk_dataset_label_horizons = (5, 10, 15)
cfg.risk_model_ensemble_size = 5
cfg.risk_model_hidden_dims = (128, 128)
cfg.risk_model_dropout = 0.10
cfg.risk_model_learning_rate = 1e-3
cfg.risk_model_batch_size = 1024
cfg.risk_model_max_epochs = 50
cfg.risk_model_patience = 8
cfg.risk_calibration_method = 'temperature'
cfg.risk_conformal_alpha = 0.10
cfg.risk_control_fail_budget = 0.20
cfg.uq_shift_suites = ('nominal_clean', 'hist_prim_shift', 'fut_prim_shift', 'hist_rmv_shift', 'high_interaction_holdout')
cfg.uq_eval_probability_bins = 15
cfg


## Step 3 - Execute Resume-Aware Benchmark Flow

In [ ]:
import pandas as pd
from src.workflows import load_existing_risk_dataset_artifact, run_uq_benchmark_flow

dataset_df = load_existing_risk_dataset_artifact(cfg.run_prefix)
if dataset_df.empty:
    raise RuntimeError('Missing persisted risk dataset artifact. Run training notebook first (or set RUN_MODE=fresh and regenerate).')

uq_bundle = run_uq_benchmark_flow(
    cfg=cfg,
    dataset_df=dataset_df,
    run_prefix=cfg.run_prefix,
    resume_mode=RUN_MODE,
)

print('loaded_from_existing =', uq_bundle.loaded_from_existing)
uq_bundle.benchmark_bundle.summary_df


In [ ]:
# Optional detail views
# uq_bundle.benchmark_bundle.per_shift_df
# uq_bundle.benchmark_bundle.reliability_df.head()
# uq_bundle.benchmark_bundle.selective_curve_df.head()


## Report-Style Notes (Fill After Run)
### Calibration
- raw vs calibrated ECE (nominal):
- raw vs calibrated ECE (shift mean):

### Robustness
- suites with largest calibration drift:
- suites with strongest calibration gains:

### Decision Utility
- selective risk behavior near deployment coverage:
